# AGEMR：SBP/DBP pooled main-effect 结果核查

目标是定位“SBP/DBP 主效应小 P 数量偏少”的环节，先核查计算与表型，不做 enrichment 或生物学解释。

1. 从各染色体原始 `*.glm.linear` 直接重算 P 阈值计数，与 pooled 汇总文件比较，并验证它们确实来自无交互 ADD 模型。
2. 查看 phenotype 的尾部、缺失、重复值、方差贡献及 SBP/DBP 配对关系；核对保存的输入与 primary-earliest 源表。
3. 每个表型选少量 SNP，把已有基因型导出到当前 workspace 的新审计子目录，按 IID 对齐，用 NumPy/SciPy 独立 OLS 复算；再用相同输入新跑一次小规模 PLINK pooled GWAS，分别对照历史结果。

**不重跑全基因组或全 1,902-SNP GWAS、不改变 phenotype、不删除或覆盖原结果。** 少量基因型中间文件保存在同一 workspace 内，不传出环境。不会打印个体 IID 或单个参与者的 BP 值。完整独立核查需要本机可执行的 PLINK2（此前已修复为 `~/plink2_env/bin/plink2`）。

样本量大本身不能决定某个稀疏 SNP 面板的显著数量；本文件将“是否有计算错误”与“现有数据中实际效应/噪声如何”分开检查。通过计算核查也不等于证明上游 phenotype 定义或原始样本 ID 映射正确。

In [ ]:
from pathlib import Path
import json, hashlib, subprocess, shutil, uuid, warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

LINEAR_RUN = None  # 可填写之前 linear run 的完整路径。
B1_ROOT = Path.home() / "workspace/agemr-data/final_cohort_v1/B1_age_varying"
PLINK_EXECUTABLE = None
RUN_INDEPENDENT_REFIT = True  # False 时只运行文件/表型/原始P值核查。
N_TOP_PER_TRAIT = 2
ADD_MEDIAN_P_VARIANT = True
TRAITS = ("BMI", "SBP", "DBP")
KEY = ["#CHROM", "POS", "ID", "REF", "ALT"]
THREADS, MEMORY_MB = 2, 1024

def require(ok,message):
    if not ok:
        raise RuntimeError(message)

if LINEAR_RUN is None:
    choices=[]
    for marker in (B1_ROOT / "B1_linear_age").glob("*/RUN_COMPLETE.json"):
        meta=json.loads(marker.read_text())
        if meta.get("status")=="complete" and all((marker.parent / "main_effect_stratified_proportions" / f"{t}_pooled_main.tsv").is_file() for t in TRAITS):
            choices.append((meta.get("finished_utc",""),marker.parent))
    require(bool(choices),"No completed linear + pooled-main results found; set LINEAR_RUN.")
    LINEAR_RUN=max(choices,key=lambda z:z[0])[1]
else:
    LINEAR_RUN=Path(LINEAR_RUN).expanduser()
completion=json.loads((LINEAR_RUN / "RUN_COMPLETE.json").read_text())
FINAL=LINEAR_RUN.parent.parent.parent
MAIN_DIR=LINEAR_RUN / "main_effect_stratified_proportions"
stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:6]
AUDIT=LINEAR_RUN / "BP_main_effect_audit" / stamp
for folder in (AUDIT,AUDIT/"qc",AUDIT/"selected_genotypes",AUDIT/"fresh_fits"):
    folder.mkdir(parents=True,exist_ok=True)
M=int(completion["panel_variants"])
print("Auditing:",LINEAR_RUN)
print("New audit directory:",AUDIT)

source_records=[]
def sha(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""):
            h.update(block)
    return h.hexdigest()

def read_keyed(path):
    x=pd.read_csv(path,sep="\t",dtype=str,keep_default_na=False)
    x["#CHROM"]=x["#CHROM"].str.replace(r"^chr","",regex=True)
    x["POS"]=pd.to_numeric(x["POS"],errors="raise").astype(int)
    require(not x.duplicated(KEY).any(),f"Duplicate variant keys: {path}")
    for col in ("P","BETA","SE","OBS_CT","A1_FREQ","T_STAT","T_OR_F_STAT"):
        if col in x:x[col]=pd.to_numeric(x[col],errors="coerce")
    source_records.append({"path":str(path),"sha256":sha(path),"bytes":Path(path).stat().st_size})
    return x

def read_iid(path):
    x=pd.read_csv(path,sep="\t",dtype=str,keep_default_na=False).rename(columns={"#IID":"IID","#FID":"FID"})
    require("IID" in x and x.IID.is_unique and x.IID.ne("").all(),f"Invalid IID column: {path}")
    if "FID" in x:
        require(x.FID.eq("0").all(),"Audit currently requires IID-only or FID=0 inputs")
        x=x.drop(columns="FID")
    return x.set_index("IID")

input_records=json.loads((LINEAR_RUN / "qc/input_files.json").read_text())
pfiles={int(Path(r["path"]).stem.replace("BMI_chr","")):Path(r["path"]).with_suffix("")
        for r in input_records if Path(r["path"]).suffix==".pgen"}
MAIN={}
RAW={}

## 1. 原始 PLINK P 值、命令与汇总计数

只统计有效 ADD 行；所有阈值严格使用 `<`。对 raw GLM 与 pooled 表比较 P/BETA/SE/OBS_CT，另存匹配差异。完整命令记录如含 `interaction`、`--parameters` 或 `--tests`，会标记供检查。此处“同一数量”只能定位汇总是否一致，不能代替后面的独立回归。

In [ ]:
counts=[]
file_audits=[]
command_audits=[]
for trait in TRAITS:
    aggregate=read_keyed(MAIN_DIR / f"{trait}_pooled_main.tsv")
    require(aggregate.TEST.eq("ADD").all(),f"{trait}: aggregate is not pooled ADD results")
    MAIN[trait]=aggregate
    raw_parts=[]
    for chrom in completion["chromosomes"]:
        prefix=MAIN_DIR / "pooled_main" / trait / f"{trait}_chr{chrom}"
        raw=read_keyed(Path(str(prefix)+f".{trait}.glm.linear"))
        require(raw.TEST.eq("ADD").all(),f"{trait} chr{chrom}: unexpected TEST")
        raw_parts.append(raw)
        marker=Path(str(prefix)+".successful_signature.json")
        status="signature_missing"
        flags=""
        if marker.is_file():
            signature=json.loads(marker.read_text())
            cmd=signature["command"]
            pheno_name=cmd[cmd.index("--pheno-name")+1] if "--pheno-name" in cmd else None
            mods=cmd[cmd.index("--glm")+1:] if "--glm" in cmd else []
            suspect=[v for v in ("interaction","--parameters","--tests") if v in cmd]
            status="pooled_command_ok" if pheno_name==trait and not suspect else "unexpected_command"
            flags=",".join(suspect)
        command_audits.append({"trait":trait,"chr":chrom,"command_status":status,"suspect_flags":flags})
    raw=pd.concat(raw_parts,ignore_index=True)
    require(not raw.duplicated(KEY).any(),f"{trait}: duplicate raw keys")
    RAW[trait]=raw
    paired=aggregate[KEY+["P","BETA","SE","OBS_CT"]].merge(raw[KEY+["P","BETA","SE","OBS_CT"]],
                  on=KEY,how="outer",validate="one_to_one",suffixes=("_aggregate","_raw"),indicator=True)
    for col in ("P","BETA","SE","OBS_CT"):
        paired[f"match_{col}"]=np.isclose(paired[col+"_aggregate"],paired[col+"_raw"],rtol=1e-10,atol=0,equal_nan=True)
    paired.to_csv(AUDIT / "qc" / f"{trait}_aggregate_vs_raw.tsv",sep="\t",index=False)
    file_audits.append({"trait":trait,"N_aggregate":len(aggregate),"N_raw":len(raw),
                       "N_key_mismatch":int(paired["_merge"].ne("both").sum()),
                       **{f"N_{col}_mismatch":int((~paired[f"match_{col}"]).sum()) for col in ("P","BETA","SE","OBS_CT")}})
    for label,table in (("aggregate",aggregate),("raw_chromosome_GLM",raw)):
        valid=table.ERRCODE.eq(".") & table.P.between(0,1)
        row={"trait":trait,"source":label,"N_rows":len(table),"N_valid":int(valid.sum()),
             "min_P":table.loc[valid,"P"].min(),"median_N":table.loc[valid,"OBS_CT"].median()}
        for threshold,name in [(0.05,"P_lt_005"),(1e-3,"P_lt_1e3"),(1e-5,"P_lt_1e5"),(5e-8,"P_lt_5e8")]:
            row[name]=int((valid & table.P.lt(threshold)).sum())
        counts.append(row)
    top=aggregate.loc[aggregate.ERRCODE.eq(".") & aggregate.P.between(0,1)].sort_values("P").head(20)
    top.to_csv(AUDIT/f"{trait}_top20_pooled_main.tsv",sep="\t",index=False)
    print("\n",trait,"top pooled associations")
    display(top[[c for c in ["ID","A1","A1_FREQ","BETA","SE","P","OBS_CT"] if c in top]])
counts=pd.DataFrame(counts)
file_audits=pd.DataFrame(file_audits)
command_audits=pd.DataFrame(command_audits)
counts.to_csv(AUDIT/"01_main_P_threshold_counts.tsv",sep="\t",index=False)
file_audits.to_csv(AUDIT/"qc/aggregate_raw_matches.tsv",sep="\t",index=False)
command_audits.to_csv(AUDIT/"qc/pooled_commands.tsv",sep="\t",index=False)
print("Raw vs aggregate threshold counts:")
display(counts)
display(file_audits)

## 2. 表型、尾部与 nuisance 调整检查

显示聚合统计而非参与者记录。极端值、异常编码或选择规则可能增加残差噪声；这里只定位，不自动删除、截尾或重定义 phenotype。`top_0.1pct_share_squared_deviation` 表示最极端约 0.1% 观测对总离均差平方和的贡献。统一线性换单位本身会同时缩放 BETA 与 SE，不能单独解释 P 值改变。

BMI 作为同一 pipeline 的对照。saved phenotype 与 primary-earliest 表逐 IID、逐数值比较；即使它们一致，也不证明原始 EHR 清洗或 person_id→genotype IID 转换已正确。

In [ ]:
def numeric_frame(x,label):
    a=x.apply(pd.to_numeric,errors="raise").astype(float)
    require(np.isfinite(a.to_numpy()).all(),f"{label}: nonfinite saved inputs")
    return a

def tail_share(values):
    a=np.asarray(values,dtype=float)
    ss=(a-a.mean())**2
    n=max(1,int(np.ceil(len(a)*0.001)))
    return float(np.sort(ss)[-n:].sum()/ss.sum()) if ss.sum()>0 else np.nan

DATA={}
phen_rows=[]
source_checks=[]
for trait in TRAITS:
    pheno_path=LINEAR_RUN / "inputs" / f"{trait}.pheno.tsv"
    covar_path=LINEAR_RUN / "inputs" / f"{trait}.covar.tsv"
    y=numeric_frame(read_iid(pheno_path)[[trait]],trait)[trait]
    cv=numeric_frame(read_iid(covar_path),f"{trait} covariates")
    require(set(y.index)==set(cv.index),f"{trait}: pheno/covar IDs differ")
    cv=cv.loc[y.index]
    meta=json.loads((LINEAR_RUN / "inputs" / f"{trait}.meta.json").read_text())
    require(sha(pheno_path)==meta["pheno_sha256"] and sha(covar_path)==meta["covar_sha256"],f"{trait}: saved inputs changed")
    a=y.to_numpy()
    W=np.column_stack([np.ones(len(y)),cv.to_numpy()])
    fit,_,rank,_=np.linalg.lstsq(W,a,rcond=None)
    resid=a-W@fit
    total=((a-a.mean())**2).sum()
    row={"trait":trait,"N":len(a),"mean":a.mean(),"sd":a.std(ddof=1),"min":a.min(),"max":a.max(),
         "n_unique":y.nunique(),"nuisance_rank":rank,"nuisance_columns_with_intercept":W.shape[1],
         "nuisance_R2":1-(resid@resid)/total if total>0 else np.nan,
         "residual_SD":np.sqrt((resid@resid)/(len(a)-rank)),
         "top_0.1pct_share_squared_deviation":tail_share(a),
         "top_0.1pct_share_residual_SSE":tail_share(resid)}
    for q in (0.0001,0.001,0.01,0.5,0.99,0.999,0.9999):row[f"q_{q:g}"]=y.quantile(q)
    phen_rows.append(row)
    common_values=y.value_counts().head(10).rename_axis("value").rename("count").reset_index()
    common_values.to_csv(AUDIT/f"{trait}_most_frequent_values.tsv",sep="\t",index=False)
    src_path=FINAL / ("bmi_primary_earliest.tsv" if trait=="BMI" else "bp_primary_earliest.tsv")
    check={"trait":trait,"primary_source":str(src_path),"source_exists":src_path.is_file()}
    if src_path.is_file():
        src=read_iid(src_path)
        common=y.index.intersection(src.index)
        values=pd.to_numeric(src.loc[common,trait],errors="coerce")
        check.update({"N_missing_from_primary":len(y)-len(common),
                      "N_value_mismatch_primary":int((~np.isclose(y.loc[common],values,rtol=1e-10,atol=1e-10,equal_nan=False)).sum())})
        del src
    source_checks.append(check)
    DATA[trait]={"y":y,"cov":cv}
    del W,resid,a
phen_summary=pd.DataFrame(phen_rows)
phen_summary.to_csv(AUDIT/"02_phenotype_distribution_and_residuals.tsv",sep="\t",index=False)
pd.DataFrame(source_checks).to_csv(AUDIT/"qc/primary_source_matches.tsv",sep="\t",index=False)
display(phen_summary)
display(pd.DataFrame(source_checks))
bp=pd.concat([DATA["SBP"]["y"],DATA["DBP"]["y"]],axis=1,join="inner")
bp_summary=pd.DataFrame([{"N_paired":len(bp),"correlation_SBP_DBP":bp.SBP.corr(bp.DBP),
                         "fraction_exactly_equal":float(bp.SBP.eq(bp.DBP).mean()),
                         "fraction_SBP_le_DBP":float(bp.SBP.le(bp.DBP).mean()),
                         "median_SBP_minus_DBP":float((bp.SBP-bp.DBP).median())}])
bp_summary.to_csv(AUDIT/"02_SBP_DBP_pair_checks.tsv",sep="\t",index=False)
display(bp_summary)

## 3. 少量 SNP 的独立 OLS 与 fresh PLINK 复核

每个 trait 选择最小 pooled P 的两个有效 SNP，加一个中位 P 的常见 SNP。选择是为了检测不同结果位置的实现一致性，不是为了重新发现信号。

导出时明确计数 ALT allele；对照原 GLM A1 后必要时仅翻转 beta 符号。按每个 SNP 的 genotype 缺失模式保留完整样本，不做均值填补。FULL 使用保存的全 nuisance block，另外计算 UNADJUSTED 与 AGE_SEX_ONLY 作为诊断；后两者不替代主模型，也不能据其显著性宣布发现。PLINK 的 dosage 与 additive export 定义以[官方文档](https://www.cog-genomics.org/plink/2.0/data#export)及本 notebook 测试为依据。

In [ ]:
def checked_run(command,prefix):
    prefix=Path(prefix)
    prefix.parent.mkdir(parents=True,exist_ok=True)
    Path(str(prefix)+".command.json").write_text(json.dumps(command,indent=2))
    with Path(str(prefix)+".stdout.txt").open("w") as out,Path(str(prefix)+".stderr.txt").open("w") as err:
        r=subprocess.run(command,stdout=out,stderr=err)
    if r.returncode:
        detail=Path(str(prefix)+".stderr.txt").read_text(errors="replace")[-2000:]
        raise RuntimeError(f"PLINK return code {r.returncode}; {prefix}\n{detail}")

def ols_one(y,cov,g):
    keep=np.isfinite(g) & np.isfinite(y) & np.isfinite(cov).all(axis=1)
    yy=y[keep];gg=g[keep]
    X=np.column_stack([np.ones(keep.sum()),cov[keep],gg])
    beta,_,rank,_=np.linalg.lstsq(X,yy,rcond=None)
    if rank!=X.shape[1] or len(yy)<=rank:
        return {"N":int(keep.sum()),"status":"rank_deficient_or_insufficient_N"}
    df=len(yy)-rank
    residual=yy-X@beta
    sigma2=float(residual@residual/df)
    se=np.sqrt(np.linalg.inv(X.T@X)[-1,-1]*sigma2)
    t=beta[-1]/se
    return {"N":int(keep.sum()),"status":"ok","BETA_ALT":beta[-1],"SE":se,"T":t,
            "P":float(2*stats.t.sf(abs(t),df)),"DF_RESID":int(df),"residual_SD":np.sqrt(sigma2)}

if RUN_INDEPENDENT_REFIT:
    candidates=[PLINK_EXECUTABLE,str(Path.home()/"plink2_env/bin/plink2"),shutil.which("plink2")]
    PLINK=None
    for candidate in candidates:
        if not candidate or not Path(candidate).is_file():continue
        try:r=subprocess.run([candidate,"--version"],capture_output=True,text=True,timeout=30)
        except (OSError,subprocess.TimeoutExpired):continue
        if r.returncode==0 and "PLINK" in (r.stdout+r.stderr):
            PLINK=candidate
            print((r.stdout+r.stderr).strip())
            break
    require(PLINK is not None,"No executable PLINK2; set PLINK_EXECUTABLE. Earlier audit tables have already been saved.")
    selected={}
    selections=[]
    for trait in TRAITS:
        tab=MAIN[trait]
        valid=tab.ERRCODE.eq(".") & tab.P.between(0,1) & tab.SE.gt(0)
        ordered=tab.loc[valid].sort_values("P")
        chosen=ordered.head(N_TOP_PER_TRAIT)
        if ADD_MEDIAN_P_VARIANT:
            common=ordered.loc[ordered.A1_FREQ.between(.05,.95)] if "A1_FREQ" in ordered else ordered
            if len(common):chosen=pd.concat([chosen,common.iloc[[len(common)//2]]]).drop_duplicates(KEY)
        require(len(chosen)>0,f"{trait}: no valid main SNPs to audit")
        selected[trait]=chosen
        selections.append(chosen[KEY].assign(trait=trait))
    selections=pd.concat(selections,ignore_index=True)
    selections.to_csv(AUDIT/"03_selected_variants.tsv",sep="\t",index=False)
    all_selected=selections[KEY].drop_duplicates()
    GENOTYPES=[]
    for chrom,subset in all_selected.groupby("#CHROM",sort=False):
        prefix=AUDIT/"selected_genotypes"/f"chr{chrom}"
        extract=Path(str(prefix)+".extract.txt")
        alleles=Path(str(prefix)+".count_ALT.tsv")
        subset.ID.to_csv(extract,index=False,header=False)
        subset[["ID","ALT"]].to_csv(alleles,sep="\t",index=False,header=False)
        command=[PLINK,"--pfile",str(pfiles[int(chrom)]),"--extract",str(extract),
                 "--export","A","--export-allele",str(alleles),
                 "--threads",str(THREADS),"--memory",str(MEMORY_MB),"--out",str(prefix)]
        checked_run(command,prefix)
        raw=pd.read_csv(Path(str(prefix)+".raw"),sep=r"\s+",dtype={"FID":str,"IID":str})
        require(raw.IID.is_unique,"Duplicate IID in genotype export")
        if "FID" in raw:require(raw.FID.eq("0").all(),"Nonzero FID requires explicit mapping")
        raw=raw.set_index("IID")
        cols={f"{row.ID}_{row.ALT}":row.ID for row in subset.itertuples(index=False)}
        require(set(cols).issubset(raw),f"Unexpected export columns; cannot verify ALT orientation: {list(raw.columns)}")
        GENOTYPES.append(raw[list(cols)].rename(columns=cols).apply(pd.to_numeric,errors="coerce"))
    G=pd.concat(GENOTYPES,axis=1,join="outer")
    require(G.columns.is_unique,"Duplicate exported genotype columns")
    results=[]
    for trait in TRAITS:
        y=DATA[trait]["y"];cov=DATA[trait]["cov"]
        require(y.index.isin(G.index).all(),f"{trait}: phenotype samples missing from export")
        chosen=selected[trait]
        fresh_parts=[]
        for chrom,subset in chosen.groupby("#CHROM",sort=False):
            prefix=AUDIT/"fresh_fits"/f"{trait}_chr{chrom}"
            extract=Path(str(prefix)+".extract.txt");subset.ID.to_csv(extract,index=False,header=False)
            command=[PLINK,"--pfile",str(pfiles[int(chrom)]),"--extract",str(extract),
                     "--pheno",str(LINEAR_RUN/"inputs"/f"{trait}.pheno.tsv"),"--pheno-name",trait,
                     "--covar",str(LINEAR_RUN/"inputs"/f"{trait}.covar.tsv"),"--glm","hide-covar",
                     "--no-input-missing-phenotype","--threads",str(THREADS),"--memory",str(MEMORY_MB),"--out",str(prefix)]
            checked_run(command,prefix)
            fresh_parts.append(read_keyed(Path(str(prefix)+f".{trait}.glm.linear")))
        fresh=pd.concat(fresh_parts).set_index("ID",verify_integrity=True)
        for _,row in chosen.iterrows():
            g=G.loc[y.index,row.ID].to_numpy(dtype=float)
            for model,cols in [("FULL_saved_nuisance",cov.columns.tolist()),
                               ("AGE_SEX_ONLY_diagnostic",[c for c in cov if c=="AGE10" or c.startswith("SAB_")]),
                               ("UNADJUSTED_diagnostic",[])]:
                r=ols_one(y.to_numpy(),cov[cols].to_numpy(),g)
                r.update({"trait":trait,"ID":row.ID,"model":model,"ALT_counted":row.ALT,
                          "saved_A1":row.A1,"saved_BETA":row.BETA,"saved_SE":row.SE,"saved_P":row.P,"saved_N":row.OBS_CT})
                if r["status"]=="ok" and model=="FULL_saved_nuisance":
                    f=fresh.loc[row.ID]
                    require(row.A1 in (row.REF,row.ALT) and f.A1 in (row.REF,row.ALT),"Unresolved A1 orientation")
                    expected_beta=row.BETA*(1 if row.A1==row.ALT else -1)
                    fresh_beta=f.BETA*(1 if f.A1==row.ALT else -1)
                    r.update({"fresh_A1":f.A1,"fresh_BETA":f.BETA,"fresh_SE":f.SE,"fresh_P":f.P,"fresh_N":f.OBS_CT,
                              "saved_vs_OLS_N_match":r["N"]==row.OBS_CT,
                              "saved_vs_OLS_beta_SE_match":bool(np.allclose([r["BETA_ALT"],r["SE"]],[expected_beta,row.SE],rtol=5e-4,atol=1e-10)),
                              "saved_vs_OLS_P_match":bool(np.isclose(r["P"],row.P,rtol=2e-3,atol=1e-300)),
                              "fresh_vs_OLS_N_match":r["N"]==f.OBS_CT,
                              "fresh_vs_OLS_beta_SE_match":bool(np.allclose([r["BETA_ALT"],r["SE"]],[fresh_beta,f.SE],rtol=5e-4,atol=1e-10)),
                              "fresh_vs_OLS_P_match":bool(np.isclose(r["P"],f.P,rtol=2e-3,atol=1e-300))})
                results.append(r)
    independent=pd.DataFrame(results)
    independent.to_csv(AUDIT/"03_independent_OLS_and_fresh_PLINK.tsv",sep="\t",index=False)
    full_checks=independent.loc[independent.model.eq("FULL_saved_nuisance")].copy()
    full_checks.to_csv(AUDIT/"03_FULL_model_agreement.tsv",sep="\t",index=False)
    display(full_checks)
    print("Diagnostic simplified models (not replacement discovery results):")
    display(independent[[c for c in ["trait","ID","model","N","BETA_ALT","SE","P","residual_SD"] if c in independent]])
else:
    print("Independent genotype/OLS/fresh-PLINK checks skipped by configuration.")

## 定位结果

- **原始 chr GLM 与汇总不一致：** 首先修复聚合、文件选择或缓存问题。
- **原始 GLM、fresh GLM、独立 FULL OLS 一致：** 对这些被核查 SNP，当前公式/列读取/样本顺序的计算可重复；继续核对 phenotype 清洗、ID 来源、极端值、位点覆盖等上游环节，不能宣布所有数据都已正确。
- **fresh PLINK 与 OLS 一致，但历史结果不一致：** 检查旧结果对应输入、软件版本、缓存及文件修改。
- **fresh PLINK 与 OLS 不一致：** 查看 ALT/A1 定向、dosage 精度、缺失样本和协变量；在差异解决前不解释分层结果。
- **表型尾部或少量观测占据很大方差：** 回到原始有效值定义核对；本 notebook 不自动截尾或改变分析集。
- **简化模型与完整模型不同：** 只是调整敏感性，不能据此选择更显著的模型作为主结果。

最适合回传的表格是 `01_main_P_threshold_counts.tsv`、`02_phenotype_distribution_and_residuals.tsv`、`03_FULL_model_agreement.tsv`；它们均为聚合或 SNP 级结果，不需要发送个体级文件。

In [ ]:
pd.DataFrame(source_records).to_csv(AUDIT/"qc/source_files.tsv",sep="\t",index=False)
(AUDIT/"AUDIT_COMPLETE.json").write_text(json.dumps({"status":"complete","linear_run":str(LINEAR_RUN),
    "audit_dir":str(AUDIT),"independent_refit_requested":RUN_INDEPENDENT_REFIT,
    "scope":"diagnostic; does not certify upstream phenotype or individual identity correctness",
    "finished_utc":datetime.now(timezone.utc).isoformat()},indent=2)+"\n")
print("Audit finished:",AUDIT)

**交付验证：** 已在合成 workspace 完整运行，核对 raw/aggregate 一致性、三个表型输入匹配、NumPy OLS 与独立 statsmodels、fresh PLINK 的系数/标准误/P、缺失 genotype、fractional dosage 导出及 ALT-major 时 A1 翻转。另验证极端值方差贡献指标。用户真实 cohort 尚未执行；不据合成测试判定真实血压数据正确。